In [0]:
# Configuration

from pyspark.sql import functions as F
from datetime import datetime

DATABASE_NAME   = "education_db"
SILVER_TABLE    = f"{DATABASE_NAME}.silver_enrollment"
GOLD_YOY        = f"{DATABASE_NAME}.gold_yoy_trend"
GOLD_GENDER     = f"{DATABASE_NAME}.gold_gender_dist"
GOLD_DROPOUT    = f"{DATABASE_NAME}.gold_dropout_risk"
GOLD_COMPARE    = f"{DATABASE_NAME}.gold_school_comparison"
GOLD_FUNNEL     = f"{DATABASE_NAME}.gold_grade_funnel"
GOLD_KPI        = f"{DATABASE_NAME}.gold_kpi_summary"
RUN_ID          = datetime.now().strftime("%Y%m%d_%H%M%S")

spark.sql(f"USE {DATABASE_NAME}")

# Load Silver — all Gold tables read from this single clean source
df = spark.table(SILVER_TABLE)
silver_count = df.count()

print(f"Silver rows loaded: {silver_count:,}")
print(f"Run ID: {RUN_ID}")
print(f"\nWill create 6 Gold tables:")
for t in [GOLD_YOY, GOLD_GENDER, GOLD_DROPOUT, GOLD_COMPARE, GOLD_FUNNEL, GOLD_KPI]:
    print(f"  {t}")

In [0]:
# Gold Table 1: Year-over-Year Enrollment Trend
# **Business Question:** How is total enrollment and academic performance changing each year?
# Used in: Dashboard Line Chart (Enrollment Trend)

gold_yoy = df.groupBy("academic_year").agg(
    F.sum("total_enrollment")                    .alias("total_students"),
    F.sum("male_count")                          .alias("total_male"),
    F.sum("female_count")                        .alias("total_female"),
    F.round(F.avg("avg_performance_score"), 2)   .alias("avg_performance"),
    F.round(F.avg("dropout_rate") * 100, 2)      .alias("avg_dropout_pct"),
    F.round(F.avg("budget_per_student_usd"), 0)  .alias("avg_budget_usd"),
    F.countDistinct("school_name")               .alias("active_schools"),
    F.countDistinct("region")                    .alias("active_regions")
).orderBy("academic_year")

gold_yoy.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(GOLD_YOY)

print(f"Gold 1 — YoY Trend: {gold_yoy.count()} rows written to {GOLD_YOY}")
gold_yoy.display()

In [0]:
# Gender Distribution
# **Business Question:** What is the gender balance per school per year?
# Used in: Dashboard Grouped Bar Chart

gold_gender = df.groupBy("school_name", "region", "academic_year").agg(
    F.sum("male_count")                            .alias("total_male"),
    F.sum("female_count")                          .alias("total_female"),
    F.sum("total_enrollment")                      .alias("total_students"),
    F.round(F.avg("gender_ratio_female") * 100, 1) .alias("female_pct")
).orderBy("school_name", "academic_year")

gold_gender.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(GOLD_GENDER)

print(f"Gold 2 — Gender Dist: {gold_gender.count()} rows written to {GOLD_GENDER}")
gold_gender.display(10)

In [0]:
# Gold Table 3: Dropout Risk Analysis
# **Business Question:** Which school-grade combinations have dangerously high dropout rates?
# Used in: Dashboard Risk Table (conditional formatting by risk_flag)

gold_dropout = df.groupBy(
    "school_name", "region", "school_type", "grade"
).agg(
    F.round(F.avg("dropout_rate") * 100, 2).alias("avg_dropout_pct"),
    F.round(F.max("dropout_rate") * 100, 2).alias("max_dropout_pct"),
    F.sum("total_enrollment")              .alias("affected_students"),
    F.first("dropout_risk_flag")           .alias("risk_flag")
).orderBy(F.desc("avg_dropout_pct"))

gold_dropout.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(GOLD_DROPOUT)

print(f"Gold 3 — Dropout Risk: {gold_dropout.count()} rows written to {GOLD_DROPOUT}")
print("\nTop 10 highest dropout risk combinations:")
gold_dropout.display(10, truncate=False)


In [0]:
# School vs Region Comparison (PySpark JOIN)
# **Business Question:** How does each school perform relative to its regional average?
# This cell demonstrates the mandatory **PySpark JOIN** from project requirements.

# ── School-level aggregation ────────────────────────────────────────────────
school_agg = df.groupBy("school_name", "region", "school_type").agg(
    F.sum("total_enrollment")                   .alias("total_enrollment"),
    F.round(F.avg("avg_performance_score"), 2)  .alias("school_avg_score"),
    F.round(F.avg("dropout_rate") * 100, 2)     .alias("school_dropout_pct"),
    F.round(F.avg("budget_per_student_usd"), 0) .alias("avg_budget_usd")
)

# ── Regional average aggregation ────────────────────────────────────────────
region_agg = df.groupBy("region").agg(
    F.round(F.avg("avg_performance_score"), 2).alias("region_avg_score"),
    F.round(F.avg("dropout_rate") * 100, 2)   .alias("region_dropout_pct")
)

# ── PySpark INNER JOIN: school data + regional benchmark ───────────────────
gold_compare = school_agg \
    .join(region_agg, on="region", how="inner") \
    .withColumn("score_vs_region",
        F.round(F.col("school_avg_score") - F.col("region_avg_score"), 2)
    ) \
    .withColumn("performance_rank",
        F.when(F.col("score_vs_region") >  5, "Above Average")
         .when(F.col("score_vs_region") < -5, "Below Average")
         .otherwise("On Par")
    ) \
    .orderBy("region", F.desc("school_avg_score"))

gold_compare.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(GOLD_COMPARE)

print(f"Gold 4 — School Comparison (JOIN): {gold_compare.count()} rows written to {GOLD_COMPARE}")
gold_compare.display(truncate=False)


In [0]:
# Gold Table 5: Grade Enrollment Funnel
# MAGIC **Business Question:** Does student enrollment drop from Grade 1 through Grade 12?
# MAGIC Used in: Dashboard Bar Chart (Enrollment Funnel)


gold_funnel = df.groupBy("grade").agg(
    F.round(F.avg("total_enrollment"), 0)      .alias("avg_enrollment"),
    F.sum("total_enrollment")                  .alias("total_enrollment"),
    F.round(F.avg("dropout_rate") * 100, 2)    .alias("avg_dropout_pct"),
    F.round(F.avg("avg_performance_score"), 1) .alias("avg_score")
).orderBy(
    F.regexp_extract(F.col("grade"), r"(\d+)", 1).cast("int")
)

gold_funnel.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(GOLD_FUNNEL)

print(f"Gold 5 — Grade Funnel: {gold_funnel.count()} rows written to {GOLD_FUNNEL}")
gold_funnel.display()

In [0]:
# KPI Summary
# MAGIC **Business Question:** What are the headline KPIs for the entire platform?
# MAGIC Used in: Dashboard Counter Cards (top row — total students, avg score, dropout %)

gold_kpi = df.agg(
    F.sum("total_enrollment")                               .alias("total_students_enrolled"),
    F.countDistinct("school_name")                          .alias("total_schools"),
    F.countDistinct("region")                               .alias("total_regions"),
    F.round(F.avg("avg_performance_score"), 1)              .alias("overall_avg_score"),
    F.round(F.avg("dropout_rate") * 100, 2)                 .alias("overall_dropout_pct"),
    F.round(F.avg("budget_per_student_usd"), 0)             .alias("avg_budget_usd"),
    F.sum(F.when(F.col("dropout_risk_flag") == "HIGH", 1)
           .otherwise(0))                                   .alias("high_risk_count"),
    F.sum(F.when(F.col("dropout_risk_flag") == "MEDIUM", 1)
           .otherwise(0))                                   .alias("medium_risk_count"),
    F.sum(F.when(F.col("performance_band") == "Excellent", 1)
           .otherwise(0))                                   .alias("excellent_count"),
    F.current_timestamp()                                   .alias("computed_at")
)

gold_kpi.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(GOLD_KPI)

print(f"Gold 6 — KPI Summary written to {GOLD_KPI}")
gold_kpi.display(truncate=False)

In [0]:
# Final Summary: All 6 Gold Tables

print("=" * 65)
print("GOLD LAYER COMPLETE — ALL 6 TABLES WRITTEN")
print("=" * 65)

tables = {
    "YoY Enrollment Trend":    GOLD_YOY,
    "Gender Distribution":     GOLD_GENDER,
    "Dropout Risk Analysis":   GOLD_DROPOUT,
    "School vs Region (JOIN)": GOLD_COMPARE,
    "Grade Enrollment Funnel": GOLD_FUNNEL,
    "KPI Summary":             GOLD_KPI,
}

for label, tname in tables.items():
    count = spark.table(tname).count()
    print(f"  {label:<30} {tname:<45} {count:>5} rows")

print("=" * 65)
print(f"\nAll tables are now available in Databricks SQL for dashboarding.")
print(f"Run ID: {RUN_ID}")

In [0]:
# SQL Preview: Ready for Dashboard

# MAGIC %md
# MAGIC ### Dashboard Query 1 — Enrollment Trend


# MAGIC %sql
# MAGIC SELECT academic_year, total_students, avg_performance, avg_dropout_pct, active_schools
# MAGIC FROM education_db.gold_yoy_trend
# MAGIC ORDER BY academic_year;


# MAGIC %md
# MAGIC ### Dashboard Query 2 — Top Dropout Risk Schools


# MAGIC %sql
# MAGIC SELECT school_name, region, grade, avg_dropout_pct, affected_students, risk_flag
# MAGIC FROM education_db.gold_dropout_risk
# MAGIC WHERE risk_flag = 'HIGH'
# MAGIC ORDER BY avg_dropout_pct DESC
# MAGIC LIMIT 15;

# MAGIC %md
# MAGIC ### Dashboard Query 3 — School vs Region Comparison

# MAGIC %sql
# MAGIC SELECT school_name, region, school_avg_score, region_avg_score,
# MAGIC        score_vs_region, performance_rank
# MAGIC FROM education_db.gold_school_comparison
# MAGIC ORDER BY region, school_avg_score DESC;

# MAGIC %md
# MAGIC ### Dashboard Query 4 — Headline KPIs

# MAGIC %sql
# MAGIC SELECT
# MAGIC   total_students_enrolled,
# MAGIC   total_schools,
# MAGIC   overall_avg_score,
# MAGIC   overall_dropout_pct,
# MAGIC   high_risk_count,
# MAGIC   avg_budget_usd
# MAGIC FROM education_db.gold_kpi_summary;

# COMMAND ----------

# MAGIC %md
# MAGIC ### Dashboard Query 5 — Grade Funnel

# COMMAND ----------

# MAGIC %sql
# MAGIC SELECT grade, avg_enrollment, avg_dropout_pct, avg_score
# MAGIC FROM education_db.gold_grade_funnel
# MAGIC ORDER BY CAST(regexp_extract(grade, '(\\d+)', 1) AS INT);